# Desikan–Killiany–Tourville parcellation

Compute DKT anatomical labels and the optional lobar parcellation from a T1 MRI.

**Resources:** downloads a sample T1, templates and pretrained weights on first use.
CPU is the default; allow substantial memory and several minutes or longer.
Set `t1_path` to an absolute path to use your own scalar 3-D T1 MRI.

Install ANTsTorch and notebook tools from the repository root before opening this notebook:
```bash
python -m pip install -e '.[notebooks]'
python -m ipykernel install --user --name python3 --display-name "Python 3 (ANTsTorch)"
```
Select that kernel and run the cells in order. All examples default to CPU.

In [ ]:
import torch  # Import before ANTs to avoid OpenMP conflicts on macOS.
import numpy as np
import matplotlib.pyplot as plt
import ants
import antstorch

device = "cpu"
antstorch.set_default_device(device)

## Load the sample MRI or choose a local image

In [ ]:
from pathlib import Path

# Set an absolute path to use your own scalar 3-D T1 MRI instead.
t1_path = None
if t1_path is None:
    t1_path = antstorch.get_antstorch_data("mprage_hippmapp3r")
t1 = ants.image_read(str(t1_path)).clone("float")
assert t1.dimension == 3 and not t1.has_components
assert np.isfinite(t1.numpy()).all()
print("Input:", t1_path, "shape:", t1.shape, "spacing:", t1.spacing)

## Request anatomical and lobar labels
Enabling `do_lobar_parcellation` returns a dictionary and runs additional tissue
segmentation internally. Without lobar parcellation or probabilities, this API
returns a single ANTsImage instead. Probabilities are disabled to limit memory.

In [ ]:
result = antstorch.desikan_killiany_tourville_labeling(
    t1, do_preprocessing=True, do_lobar_parcellation=True,
    return_probability_images=False, backend="antstorch",
    device=device, verbose=True,
)
segmentation = result["segmentation_image"]
lobes = result["lobar_parcellation"]

values = segmentation.numpy()
assert segmentation.shape == t1.shape
assert np.isfinite(values).all()
np.testing.assert_allclose(values, np.rint(values), atol=1e-6)
np.testing.assert_allclose(segmentation.spacing, t1.spacing)
np.testing.assert_allclose(segmentation.origin, t1.origin)
np.testing.assert_allclose(segmentation.direction, t1.direction)
assert (values > 0).any()

In [ ]:
assert lobes.shape == t1.shape
assert np.isfinite(lobes.numpy()).all()
np.testing.assert_allclose(lobes.spacing, t1.spacing)
np.testing.assert_allclose(lobes.origin, t1.origin)
np.testing.assert_allclose(lobes.direction, t1.direction)
for image in (segmentation, lobes):
    ants.plot(t1, overlay=image, axis=2, nslices=6,
              overlay_cmap="jet", overlay_alpha=0.5)
output_dir = Path("example_outputs/desikan_killiany_tourville")
output_dir.mkdir(parents=True, exist_ok=True)
ants.image_write(segmentation, str(output_dir / "dkt_labels.nii.gz"))
ants.image_write(lobes, str(output_dir / "lobar_parcellation.nii.gz"))
print("DKT labels present:", np.unique(values).astype(int))
print("Outputs (matching files are overwritten):", output_dir.resolve())

Inspect the cortical boundaries, subcortical regions and hemisphere assignments before computing regional summaries.